In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

import torch
import torch.nn as nn
import torch.distributions as D

import math

from torch.utils.data import DataLoader

import seaborn as sns

In [ ]:
# 1. Define the mixture weights (probabilities for each Gaussian)
# This chooses between the 2 components. They must sum to 1 (or will be normalized).
mix_weights = torch.tensor([0.3, 0.7])
mix_dist = D.Categorical(probs=mix_weights)

# 2. Define the parameters for the two 2D Gaussians
# Shape format: [num_components, num_dimensions]
means = torch.tensor([
    [0.0, 0.0],   # Mean of 1st Gaussian
    [2.0, 2.0]   # Mean of 2nd Gaussian
])

s = 0.1
# Define full 2D covariance matrices for each component
# Shape format: [num_components, num_dimensions, num_dimensions]
covariances = torch.tensor([
    [[s, 0.0],   # Covariance matrix of 1st Gaussian
     [0.0, s]],
    
    [[s, 0.0],  # Covariance matrix of 2nd Gaussian
     [0.0, s]]
])

# Create the component multivariate distributions
component_dist = D.MultivariateNormal(loc=means, covariance_matrix=covariances)

# 3. Combine into a Mixture distribution
gmm = D.MixtureSameFamily(mix_dist, component_dist)

X = gmm.sample((10000,))
print("Generated 2D Samples:\n", X.shape)

counts, xedges, yedges, im = plt.hist2d(X[:,0], X[:,1], bins=50, cmap='Reds')
    

In [ ]:
sigma_list = torch.tensor([0.1, 0.5, 1])
n_sigma = len(sigma_list)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12, 4))

sigma = sigma_list[0]
X_sigma = X + sigma * torch.randn_like(X)
counts, xedges, yedges, im = ax[0].hist2d(X_sigma[:,0], X_sigma[:,1], bins=50, cmap='Reds')
ax[0].set_xlim(-1.5,3.5)
ax[0].set_ylim(-2,4)

sigma = sigma_list[1]
X_sigma = X + sigma * torch.randn_like(X)
counts, xedges, yedges, im =ax[1].hist2d(X_sigma[:,0], X_sigma[:,1], bins=50, cmap='Reds')
ax[1].set_xlim(-1.5,3.5)
ax[1].set_ylim(-2,4)

sigma = sigma_list[2]
X_sigma = X + sigma * torch.randn_like(X)
counts, xedges, yedges, im = ax[2].hist2d(X_sigma[:,0], X_sigma[:,1], bins=50, cmap='Reds')

ax[2].set_xlim(-1.5,3.5)
ax[2].set_ylim(-2,4)

In [ ]:
class MyScore(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(3, 50),
            nn.Tanh(),
            nn.Linear(50, 50), 
            nn.Tanh(),                      
            nn.Linear(50, 50),             
            nn.Tanh(),            
            nn.Linear(50, 2), 
       )
            
    def forward(self, x, sigma):
        
        # combine x and t into one tensor    
        state = torch.cat((x, sigma), dim=1)
        
        # pass input to the network
        output = self.net(state)
        
        return output
    
model = MyScore() 

In [ ]:
# batch-size
batch_size = 2000

# total training epochs
total_epochs = 1000

# Adam
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# define a dataloader
data_loader = DataLoader(X, batch_size=batch_size, shuffle=True, drop_last=True)

loss_list = []

for epoch in range(total_epochs):   # for each epoch
    
    for idx, data in enumerate(data_loader):  # loop over all mini-batches 

        i = torch.randint(0, n_sigma, size=(data.shape[0], ))

        # generate standard Gaussian random variables
        z = torch.randn_like(data) 

        sigma = sigma_list[i].reshape(-1,1) 
        
        # get samples distributed according to the Gaussian density at time t
        x_tilde = data + sigma * z
        
        # evaluate the model
        score = model(x_tilde, sigma) 

        loss = torch.mean((0.5*torch.sum(score**2, dim=1, keepdim=True) + torch.sum(score * z, dim=1, keepdim=True) / sigma) * sigma**2)
                        
        optimizer.zero_grad()
        # gradient step
        loss.backward()
        
        # update weights
        optimizer.step()
        
        if idx == 0:
            # record the loss    
            loss_list.append(loss.item())  
            if epoch % 200 == 0:
                print ('epoch=%d\n   loss=%.4f' % (epoch, loss.item()))   
                
fig, ax = plt.subplots(1,1, figsize=(5, 4))

ax.plot(loss_list)
ax.set_xlabel('epoch')
ax.set_title('loss vs epoch')             

In [ ]:
nx = 20
x = torch.linspace(-1, 3, nx)
y = torch.linspace(-1, 3, nx)
xx, yy = torch.meshgrid(x, y, indexing='ij')

states = torch.stack((xx.flatten(), yy.flatten()), dim=1)

fig, axes = plt.subplots(2, 2, figsize=(10,10))

for i in range(4):
    if i == 0 :
        sigma_val = 0.0
        ax = axes[0,0]
    else :
        sigma_val = sigma_list[i-1]

        if i == 1 :
            ax = axes[0,1]
        if i == 2 :
            ax = axes[1,0]
        if i == 3 :
            ax = axes[1,1]

    sigma = torch.ones(states.shape[0], 1) * sigma_val
    score = model(states, sigma).reshape(nx,nx,2).detach().numpy()
    
    im = ax.quiver(xx.numpy(), yy.numpy(), score[:,:,0], score[:,:,1]) 
    ax.set_xlabel(r'x',fontsize=15)

    if i == 0 or i == 2:
        ax.set_ylabel(r'y',fontsize=15)
    
    ax.set_title(f'score function, sigma={sigma_val: .1f}',fontsize=15)


Questions:
* In this test, because the data distribution is mixture of two Gaussians, the score function $\nabla \ln p_\sigma$ has analytical solution. Can you write a function to compute the score for given (x,y) and $\sigma$.
* Compare the vector fields of the true score and the learned score. How to quantify the error? Are they similar or very different?
* What to do in order to improve the accuracy of the learned score (especially at sigma=0)? How to choose the sequence of sigma values and the embeding of the sigma in the neural networks?   